In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.base import BaseEstimator, ClassifierMixin

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("=" * 80)
# LassoNet
class LassoNetClassifier(nn.Module, BaseEstimator, ClassifierMixin):

    def __init__(self, input_dim, hidden_dims=[64, 32], output_dim=2,
                 lambda_=0.001, M=10.0, device='cpu'):
        super(LassoNetClassifier, self).__init__()
        nn.Module.__init__(self)

        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.output_dim = output_dim
        self.lambda_ = lambda_
        self.M = M
        self.device = device


        self.skip_connection = nn.Linear(input_dim, output_dim, bias=True)


        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, output_dim))

        self.feedforward = nn.Sequential(*layers)


        self.to(self.device)


        self.classes_ = None

    def forward(self, x):

        linear_out = self.skip_connection(x)


        nonlinear_out = self.feedforward(x)


        output = linear_out + nonlinear_out

        return output

    def hierarchical_penalty(self):


        skip_weight = self.skip_connection.weight  # [output_dim, input_dim]


        first_layer = None
        for name, module in self.feedforward.named_modules():
            if isinstance(module, nn.Linear):
                first_layer = module
                break

        if first_layer is None:
            return 0.0

        first_weight = first_layer.weight  # [hidden_dim, input_dim]

        # sum_i max(||W_i^(1)||_∞ - M|θ_i|, 0)
        penalty = 0.0
        for i in range(self.input_dim):
            theta_i = torch.abs(skip_weight[:, i]).max()  # |θ_i|
            w_i_norm = torch.abs(first_weight[:, i]).max()  # ||W_i^(1)||_∞

            violation = torch.relu(w_i_norm - self.M * theta_i)
            penalty += violation

        return penalty

    def get_feature_importance(self):

        skip_weight = self.skip_connection.weight.detach().cpu().numpy()
        importance = np.abs(skip_weight).sum(axis=0)
        return importance / importance.sum()

    def get_selected_features(self, threshold=1e-5):

        importance = self.get_feature_importance()
        selected = np.where(importance > threshold)[0]
        return selected

    def fit(self, X, y, X_val=None, y_val=None, epochs=200, batch_size=32, lr=0.01, verbose=True):


        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values


        self.classes_ = np.unique(y)
        label_map = {label: idx for idx, label in enumerate(self.classes_)}
        y_encoded = np.array([label_map[label] for label in y])


        X_tensor = torch.FloatTensor(X).to(self.device)
        y_tensor = torch.LongTensor(y_encoded).to(self.device)


        has_validation = X_val is not None and y_val is not None
        if has_validation:
            if isinstance(X_val, pd.DataFrame):
                X_val = X_val.values
            if isinstance(y_val, pd.Series):
                y_val = y_val.values
            y_val_encoded = np.array([label_map[label] for label in y_val])
            X_val_tensor = torch.FloatTensor(X_val).to(self.device)
            y_val_tensor = torch.LongTensor(y_val_encoded).to(self.device)


        dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)


        optimizer = optim.Adam(self.parameters(), lr=lr, weight_decay=0.01)
        criterion = nn.CrossEntropyLoss()


        total_batches = len(dataloader)


        self.train()
        for epoch in range(epochs):
            epoch_loss = 0.0
            epoch_correct = 0
            epoch_total = 0

            for batch_idx, (batch_X, batch_y) in enumerate(dataloader):
                optimizer.zero_grad()


                outputs = self(batch_X)


                ce_loss = criterion(outputs, batch_y)
                l1_loss = self.lambda_ * torch.sum(torch.abs(self.skip_connection.weight))
                hier_penalty = self.hierarchical_penalty()

                total_loss = ce_loss + l1_loss + 0.01 * hier_penalty


                total_loss.backward()


                with torch.no_grad():

                    skip_weight = self.skip_connection.weight
                    skip_weight.data = torch.sign(skip_weight) * torch.relu(
                        torch.abs(skip_weight) - self.lambda_
                    )

                optimizer.step()


                epoch_loss += total_loss.item()
                _, predicted = torch.max(outputs, 1)
                epoch_correct += (predicted == batch_y).sum().item()
                epoch_total += batch_y.size(0)


            avg_loss = epoch_loss / total_batches
            train_acc = epoch_correct / epoch_total


            if has_validation:
                self.eval()
                with torch.no_grad():
                    val_outputs = self(X_val_tensor)
                    val_loss = criterion(val_outputs, y_val_tensor).item()
                    _, val_predicted = torch.max(val_outputs, 1)
                    val_acc = (val_predicted == y_val_tensor).sum().item() / y_val_tensor.size(0)
                self.train()

                if verbose:

                    progress = '=' * 30
                    print(f'Epoch {epoch + 1}/{epochs}')
                    print(
                        f'{total_batches}/{total_batches} [{progress}] - loss: {avg_loss:.4f} - accuracy: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}')
            else:
                if verbose:

                    progress = '=' * 30
                    print(f'Epoch {epoch + 1}/{epochs}')
                    print(
                        f'{total_batches}/{total_batches} [{progress}] - loss: {avg_loss:.4f} - accuracy: {train_acc:.4f}')

        return self

    def predict_proba(self, X):

        if isinstance(X, pd.DataFrame):
            X = X.values

        self.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            outputs = self(X_tensor)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()

        return probs

    def predict(self, X):

        probs = self.predict_proba(X)
        y_pred_idx = np.argmax(probs, axis=1)
        y_pred = np.array([self.classes_[idx] for idx in y_pred_idx])
        return y_pred

    def score(self, X, y):

        y_pred = self.predict(X)
        return accuracy_score(y, y_pred)



# ####data
df = pd.read_csv('Esophageal_Dataset.csv')
target_col = 'stage_event_pathologic_stage'
df = df.dropna(subset=[target_col])
leakage_features = [
    'stage_event_tnm_categories',
    'stage_event_clinical_stage',
    'stage_event_system_version',
    'stage_event_psa',
    'stage_event_gleason_grading',
    'stage_event_ann_arbor',
    'stage_event_serum_markers',
    'stage_event_igcccg_stage',
    'stage_event_masaoka_stage',
    'vital_status',
    'days_to_death',
    'days_to_last_followup',
    'person_neoplasm_cancer_status',
    'has_new_tumor_events_information',
    'primary_pathology_number_of_lymphnodes_positive_by_he',
    'primary_pathology_number_of_lymphnodes_positive_by_ihc',
    'primary_pathology_lymph_node_metastasis_radiographic_evidence',
    'primary_pathology_lymph_node_examined_count',
    'primary_pathology_primary_lymph_node_presentation_assessment',
    'primary_pathology_residual_tumor',
    'primary_pathology_postoperative_rx_tx',
    'primary_pathology_radiation_therapy',
    'primary_pathology_planned_surgery_status',
    'primary_pathology_neoplasm_histologic_grade',
]

removed_leakage = [f for f in leakage_features if f in df.columns]
df = df.drop(columns=removed_leakage)
missing_rates = df.isnull().sum() / len(df)
high_missing = missing_rates[missing_rates > 0.8].index.tolist()
high_missing = [c for c in high_missing if c != target_col]
if high_missing:
    df = df.drop(columns=high_missing)
    print(f"{len(high_missing)} features with high missing rate have been removed")


exclude_kw = ['stage', 'unnamed', 'barcode', 'uuid', 'patient_id',
              'bcr_', 'icd_', 'project', 'has_', 'vital', 'days_to', 'diagnosis']

feature_cols = []
for col in df.columns:
    if col == target_col:
        continue
    if any(kw in col.lower() for kw in exclude_kw):
        continue
    if df[col].isnull().sum() / len(df) < 0.5:
        feature_cols.append(col)


X = df[feature_cols].copy()
y = df[target_col].copy()


# Handle missing values

for col in X.select_dtypes(include=[np.number]).columns:
    if X[col].isnull().any():
        X[col].fillna(X[col].median(), inplace=True)

for col in X.select_dtypes(include=['object']).columns:
    if X[col].isnull().any():
        mode = X[col].mode()[0] if len(X[col].mode()) > 0 else 'Unknown'
        X[col].fillna(mode, inplace=True)


# Handle missing values


for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

print(f"Feature matrix: {X.shape}")

#Feature selection

K_FEATURES = 15
selector = SelectKBest(score_func=mutual_info_classif, k=min(K_FEATURES, len(feature_cols)))
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'feature': feature_cols,
    'score': selector.scores_
}).sort_values('score', ascending=False)
print(feature_scores.head(20).to_string(index=False))

selected_mask = selector.get_support()
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

print(f"\nSelected {len(selected_features)} features:")
for i, feat in enumerate(selected_features, 1):
    score = feature_scores[feature_scores['feature'] == feat]['score'].values[0]
    print(f"  {i:2d}. {feat:<65} (Score: {score:.4f})")

X_selected = X[selected_features].copy()


# Data standardization

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_selected_scaled = scaler.fit_transform(X_selected)
X_selected_scaled = pd.DataFrame(X_selected_scaled, columns=selected_features, index=X_selected.index)



# Data splitting

class_counts = y.value_counts()
print("\nNumber of samples per stage:")
for stage, count in class_counts.items():
    print(f"  {stage}: {count} 例")

min_samples = class_counts.min()
stratify_param = None if min_samples < 2 else y

X_train, X_test, y_train, y_test = train_test_split(
    X_selected_scaled, y,
    test_size=0.3,
    random_state=42,
    stratify=stratify_param
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

print("\nTraining set distribution (before SMOTE):")
for s, c in y_train.value_counts().sort_index().items():
    print(f"  {s}: {c} cases")


# SMOTE

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"\nBefore SMOTE: {X_train.shape}")
print(f"After SMOTE: {X_train_bal.shape}")
print(f"Added samples: {X_train_bal.shape[0] - X_train.shape[0]}")

print("\nTraining set distribution (after SMOTE):")
for s, c in pd.Series(y_train_bal).value_counts().sort_index().items():
    print(f"  {s}: {c} cases")


# LassoNet

n_classes = len(np.unique(y))
lassonet_model = LassoNetClassifier(
    input_dim=X_train_bal.shape[1],
    hidden_dims=[64, 32],
    output_dim=n_classes,
    lambda_=0.001,
    M=10.0,
    device=device
)


X_train_fit, X_val_fit, y_train_fit, y_val_fit = train_test_split(
    X_train_bal, y_train_bal,
    test_size=0.2,
    random_state=42,
    stratify=y_train_bal if len(np.unique(y_train_bal)) > 1 else None
)

lassonet_model.fit(
    X_train_fit, y_train_fit,
    X_val=X_val_fit, y_val=y_val_fit,
    epochs=200,
    batch_size=64,
    lr=0.001,
    verbose=True
)


#Cross-validation

def create_lassonet_for_cv():
    return LassoNetClassifier(
        input_dim=X_selected_scaled.shape[1],
        hidden_dims=[64, 32],
        output_dim=n_classes,
        lambda_=0.001,
        M=10.0,
        device=device
    )

cv_scores = []
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(kf.split(X_selected_scaled), 1):
    X_fold_train = X_selected_scaled.iloc[train_idx]
    y_fold_train = y.iloc[train_idx]
    X_fold_val = X_selected_scaled.iloc[val_idx]
    y_fold_val = y.iloc[val_idx]

    model_cv = create_lassonet_for_cv()
    model_cv.fit(X_fold_train, y_fold_train, epochs=200, batch_size=32, lr=0.01, verbose=False)

    score = model_cv.score(X_fold_val, y_fold_val)
    cv_scores.append(score)
    print(f"  Fold {fold}: {score:.4f}")

cv_scores = np.array(cv_scores)
print(f"\naverage: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")


# Feature selection analysis

feature_importance = lassonet_model.get_feature_importance()
selected_by_lassonet = lassonet_model.get_selected_features(threshold=1e-5)

lassonet_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': feature_importance
}).sort_values('importance', ascending=False)


print(lassonet_importance.to_string(index=False))

print(f"\nNumber of non-zero features: {len(selected_by_lassonet)}/{len(selected_features)}")
if len(selected_by_lassonet) > 0:
    print(f"\nFeature indices: {selected_by_lassonet.tolist()}")
    print(f"\nFeature names:")
    for idx in selected_by_lassonet:
        print(f"  - {selected_features[idx]}")


# Model evaluation

y_train_pred = lassonet_model.predict(X_train)


train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_recall = recall_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_f1 = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)


y_test_pred = lassonet_model.predict(X_test)


test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_recall = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

print("=" * 70)
print(" " * 20 + "Model Performance Comparison")
print("=" * 70)
print(f"{'Metric':<15} {'Training Set':>15} {'Test Set':>15} {'Difference':>15}")
print("-" * 70)
print(f"{'Accuracy':<15} {train_accuracy:>15.4f} {test_accuracy:>15.4f} {abs(train_accuracy - test_accuracy):>15.4f}")
print(
    f"{'Precision':<15} {train_precision:>15.4f} {test_precision:>15.4f} {abs(train_precision - test_precision):>15.4f}")
print(f"{'Recall':<15} {train_recall:>15.4f} {test_recall:>15.4f} {abs(train_recall - test_recall):>15.4f}")
print(f"{'F1 Score':<15} {train_f1:>15.4f} {test_f1:>15.4f} {abs(train_f1 - test_f1):>15.4f}")
print("=" * 70)

print(classification_report(y_test, y_test_pred, zero_division=0))
